# Evaluating Models
Companion to the evaluation notes.

## 1. Regression metrics: same model, different stories

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error

y_true_reg = np.array([10, 12, 15, 20, 23, 100.])
y_pred_reg = np.array([11, 11, 16, 19, 25, 40.])
print("MAE ", mean_absolute_error(y_true_reg, y_pred_reg))
print("RMSE", np.sqrt(mean_squared_error(y_true_reg, y_pred_reg)).round(3))
print("R2  ", round(r2_score(y_true_reg, y_pred_reg), 3))
print("MAPE", round(mean_absolute_percentage_error(y_true_reg, y_pred_reg), 3))

## 2. The ten-observation example by hand, checked

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score, roc_auc_score

y = np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0])
p = np.array([0.90, 0.80, 0.60, 0.40, 0.70, 0.30, 0.20, 0.20, 0.10, 0.05])
for c in [0.50, 0.35]:
    yp = (p >= c).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yp).ravel()
    print(f"cutoff {c}: TP={tp} FP={fp} TN={tn} FN={fn}  acc={accuracy_score(y, yp):.3f} "
          f"prec={precision_score(y, yp):.3f} rec={recall_score(y, yp):.3f} F1={f1_score(y, yp):.3f} KS={tp/(tp+fn) - fp/(fp+tn):.3f}")

pairs = [(pp > pn) for pp in p[y == 1] for pn in p[y == 0]]
print("AUC by counting pairs:", sum(pairs), "/", len(pairs), "=", round(np.mean(pairs), 4), " sklearn:", round(roc_auc_score(y, p), 4))

## 3. ROC of an ideal score, and the same score with some labels flipped

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

rng = np.random.default_rng(0)
score = rng.random(500)
ideal = (score > 0.3).astype(int)
noisy = ideal.copy()
flip = rng.choice(500, 50, replace=False)
noisy[flip] = 1 - noisy[flip]
for labels, name in [(ideal, "ideal"), (noisy, "10% labels flipped")]:
    fpr, tpr, _ = roc_curve(labels, score)
    plt.plot(fpr, tpr, label=f"{name} (AUC {roc_auc_score(labels, score):.3f})")
plt.plot([0, 1], [0, 1], "k--"); plt.xlabel("FPR"); plt.ylabel("TPR"); plt.legend()

## 4. The bank model: sweeping cutoffs and the KS cutoff

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve, auc, confusion_matrix, precision_score, recall_score

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# 1. The bank data: predict high-revenue customers (10.7% of all customers)
bd = pd.read_csv("../../data/bd_train.csv").drop(columns=["REF_NO", "post_code", "post_area"])
y = (bd.pop("Revenue.Grid") == 1).astype(int)
X = pd.get_dummies(bd, drop_first=True, dtype=float)
x_train, x_test, y_train, y_true = train_test_split(X, y, test_size=0.25, random_state=1, stratify=y)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(x_train, y_train)
scores = model.predict_proba(x_test)[:, 1]          # probability of being high-revenue

# 2. Sweep cutoffs to find maximum KS separation
cutoffs = np.linspace(0.01, 0.99, 197)
ks_values = []
tpr_list = []
fpr_list = []

for c in cutoffs:
    y_pred = (scores >= c).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    tpr = tp / (tp + fn)
    fpr = fp / (tn + fp)
    ks = tpr - fpr
    
    tpr_list.append(tpr)
    fpr_list.append(fpr)
    ks_values.append(ks)

best_idx = np.argmax(ks_values)
best_cutoff = cutoffs[best_idx]
max_ks = ks_values[best_idx]

print(f"Optimal KS Cutoff: {best_cutoff:.3f}")
print(f"Maximum KS Value:  {max_ks:.4f}")

# 3. Compute ROC-AUC
fpr_sk, tpr_sk, thresholds = roc_curve(y_true, scores)
roc_auc = auc(fpr_sk, tpr_sk)
print(f"Model ROC-AUC:     {roc_auc:.4f}")

# 4. Compare default cutoff (0.50) vs optimal KS cutoff
y_pred_default = (scores >= 0.50).astype(int)
y_pred_optimal = (scores >= best_cutoff).astype(int)

print("\n--- At Default Cutoff (0.50) ---")
print(f"Precision: {precision_score(y_true, y_pred_default):.4f}")
print(f"Recall:    {recall_score(y_true, y_pred_default):.4f}")

print(f"\n--- At Optimal KS Cutoff ({best_cutoff:.3f}) ---")
print(f"Precision: {precision_score(y_true, y_pred_optimal):.4f}")
print(f"Recall:    {recall_score(y_true, y_pred_optimal):.4f}")

## 5. ROC and precision-recall curves

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

fpr, tpr, thr = roc_curve(y_true, scores)
prec, rec, _ = precision_recall_curve(y_true, scores)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(fpr, tpr); ax[0].plot([0, 1], [0, 1], "k--"); ax[0].set_title(f"ROC, AUC {roc_auc_score(y_true, scores):.3f}")
ax[1].plot(rec, prec); ax[1].axhline(y_true.mean(), color="k", ls="--"); ax[1].set_title(f"PR, AP {average_precision_score(y_true, scores):.3f}")
ax[0].set_xlabel("FPR"); ax[0].set_ylabel("TPR"); ax[1].set_xlabel("recall"); ax[1].set_ylabel("precision")

## 6. Choosing the cutoff by business cost
A missed high-revenue customer costs 10, a wasted contact costs 1.

In [ ]:
def total_cost(cutoff, cost_fn=10, cost_fp=1):
    pred = scores >= cutoff
    fn = ((~pred) & (y_true == 1)).sum()
    fp = (pred & (y_true == 0)).sum()
    return cost_fn * fn + cost_fp * fp

grid = np.linspace(0.01, 0.9, 90)
costs = [total_cost(c) for c in grid]
best = grid[int(np.argmin(costs))]
print(f"best cutoff {best:.2f}, cost {min(costs)};  cost at 0.5: {total_cost(0.5)}")
plt.plot(grid, costs); plt.axvline(best, color="red", ls="--"); plt.xlabel("cutoff"); plt.ylabel("total cost")

## 7. More than two classes

In [ ]:
from sklearn.metrics import classification_report

y_mc = np.array(["A"] * 50 + ["B"] * 40 + ["C"] * 10)
pred_mc = np.array(["A"] * 45 + ["B"] * 5 + ["B"] * 36 + ["A"] * 4 + ["C"] * 2 + ["A"] * 8)
print(classification_report(y_mc, pred_mc, digits=3))

## Try it yourself
1. Metrics at cutoff 0.75 in section 2.
2. Instance 5's score 0.70 -> 0.45: recompute the AUC by counting pairs.
3. Cost ratio 3:1 in section 6: which way does the best cutoff move?
4. Last regression prediction 95 instead of 40: MAE and RMSE?